# Soldani - Second task - Benchmark

## Context: how this experiment evolved

This notebook is kept in its original, unpolished form on purpose. It documents a real, imperfect first attempt at comparing FairMind against an LLM, and the mistakes recorded here are exactly what motivated the corrections applied in the later benchmark notebooks (see `2_3_benchmark_thor.ipynb`). Nothing below has been cleaned up to hide what went wrong.

**The setup.** The supervisor initially provided a personal OpenAI API key for these experiments: the model used at that stage was `o3-mini` with `reasoning={"effort": "high"}`, which is why the earliest version of this notebook tracked `input_tokens`, `output_tokens`, *and* `reasoning_tokens` - the latter only exists for OpenAI's reasoning-capable models. Later in the project the backend was switched to a local `llama.cpp` server serving Qwen. Since that server doesn't expose OpenAI's reasoning-token accounting, `reasoning_tokens` is hardcoded to `None` in the version below.

**The row-count mistake.** In the very first version of this experiment, the prompt sent to the LLM sampled only **300 rows** of the dataset, while FairMind fitted its Bayesian Network on the full ~49,000-row training set - the LLM was seeing roughly 1/150th of what FairMind used. Once I noticed the discrepancy, I reported it to the supervisor, it was clear this alone could explain a large part of the errors observed up to that point. I raised the sample size to the 2,000 rows used below - still under 5% of the full dataset, so the two methods still weren't really comparable on equal footing.

**What actually broke this particular run.** Even with more rows, the deeper issue was that neither `education` (16 raw categories) nor `hours-per-week` (96 distinct raw values) is discretized in this notebook. As a result, the LLM must implicitly reason over as many as 16 × 96 = 1,536 possible (confounder, mediator) combinations directly from a raw CSV dump, far beyond what can realistically be tracked within a single prompt. This is consistent with the all-zero output observed below. Increasing the number of rows was therefore a necessary improvement, but not a sufficient one. The real solution, implemented in the later benchmark notebooks, was to stop asking the LLM to compute causal quantities directly from raw tabular data and instead provide it with an appropriate statistical representation of the dataset.

## 1. Initial setup

Locates the repository root by walking up from the current working directory until it finds the `src/` package, then adds it to `sys.path` so the `src.*` modules can be imported regardless of where Jupyter was launched from.

In [ ]:
# Find the root
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Imports and LLM client configuration

Imports the FairMind building blocks (`build_sfm`, `fit_discrete_bayesian_model`, the five effect functions) and sets up an OpenAI-compatible client. `MODEL_NAME` is the identifier the local `llama.cpp` server expects when routing the request to the loaded Qwen weights.

In [61]:
import json
import pandas as pd

from openai import OpenAI
from pgmpy.estimators import BayesianEstimator
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect, spurious_effect,
    natural_direct_effect, natural_indirect_effect,
)

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
)

# Original setup pointed at the supervisor's real OpenAI account
# (model="o3-mini"); switched to a local llama.cpp/Qwen server
#client = OpenAI()
MODEL_NAME = "qwen2.5-7b-instruct"

## 3. Benchmark configuration

Defines the Standard Fairness Model roles for this run on the Adult dataset: `S2_gender` as the protected attribute X (`x0`=Female, `x1`=Male), `T_income` as the outcome Y (target state `>50K`), `hours-per-week` as the mediator W, `education` as the confounder Z.

In [62]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

## 4. FairMind - exact ground truth

Builds the SFM graph, fits a Discrete Bayesian Network on the raw (non-binned) data with BDeu-smoothed parameter estimation, and computes the five causal fairness effects by exact inference. This is the reference against which the LLM's answer is compared below.

In [ ]:
import time

def run_fairmind(config: dict) -> tuple[dict, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    # The five SFM causal fairness quantities (Plecko & Bareinboim, 2024).
    tv = total_variation(bn, target, config["protected"], x0, x1)
    te = total_effect(bn, target, config["protected"], x0, x1)
    effects = {
        "TV": tv,
        "TE": te,
        # SE = TV - TE (Eq. 3, Plecko & Bareinboim 2024). This used to call
        # spurious_effect(bn, target, protected, x0) instead, a different,
        # single-argument quantity -- see the closing note at the end.
        "SE": tv - te,
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, elapsed

ground_truth, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind - elapsed time: {fairmind_time:.4f}s")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

2026-08-07 22:32:31.913 | DEBUG    | src.model:fit_discrete_bayesian_model:33 - Using estimator: <class 'pgmpy.estimators.BayesianEstimator.BayesianEstimator'> with parameters: {'prior_type': 'BDeu'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'hours-per-week': 'N', 'education': 'C', 'T_income': 'C'}
2026-08-07 22:32:32.015 | DEBUG    | src.effects:total_variation:248 - Computing total variation for target=('T_income', '>50K'), private_baseline=Female, private_mod=Male


FairMind - elapsed time: 0.0049s
  TV: 0.194470
  TE: 0.183161
  SE: 0.011309
  DE: 0.137049
  IE: -0.046112


## 5. Building the prompt for the LLM

Samples rows from the same raw dataset and writes them into the prompt as a CSV block, together with the SFM variable roles and the identification formulae - the LLM only has to apply these formulae, not derive them from scratch.

In [ ]:
def build_gpt_prompt(config: dict, n_rows: int = 2000) -> str:
    # Raised from an original 300 once I noticed FairMind was trained on
    # the full ~49k-row dataset while the LLM only ever saw a small sample.
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()
    sample = df.sample(n=min(n_rows, len(df)), random_state=42)
    csv_str = sample.to_csv(index=False)

    return f"""You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

DATASET ({len(sample)} rows):
{csv_str}

VARIABLE ROLES:
- X (protected): "{config['protected']}", x0="{config['x0']}", x1="{config['x1']}"
- Y (target):    "{config['target_col']}", target state="{config['target_val']}"
- W (mediators): {config['mediators']}
- Z (confounders): {config['confounders']}

IDENTIFICATION FORMULAE (use these exactly):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)
- SE = TV - TE
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)
- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)

Return ONLY a JSON object, no other text:
{{
  "TV": <float>,
  "TE": <float>,
  "SE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""

prompt = build_gpt_prompt(CONFIG)
print(prompt[:600], "\n[...]")

You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

DATASET (2000 rows):
S2_gender,hours-per-week,education,T_income
Female,40,HS-grad,<=50K
Male,40,HS-grad,<=50K
Female,40,Bachelors,>50K
Male,40,HS-grad,<=50K
Female,30,Bachelors,<=50K
Female,40,HS-grad,<=50K
Male,45,HS-grad,<=50K
Female,40,Bachelors,>50K
Male,50,HS-grad,<=50K
Female,40,Some-college,<=50K
Male,70,Some-college,<=50K
Male,30,Bachelors,<=50K
Male,40,Some-college,<=50K
Male,45,11th,<=50K
Male,40,Bachelors,<=50K
Male,60,Assoc-voc,<=50K
Fema 
[...]


## 6. Calling the LLM and parsing its response

Sends the prompt to the configured model with `temperature=0` for determinism, records token usage, and parses the JSON block out of the response text.

In [ ]:
def call_gpt(prompt: str, model: str = MODEL_NAME) -> tuple[dict, dict, float]:
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    elapsed = time.perf_counter() - start

    usage = {
        "input_tokens":     response.usage.prompt_tokens,
        "output_tokens":    response.usage.completion_tokens,
        "reasoning_tokens": None,
        "total_tokens":     response.usage.total_tokens,
    }

    raw = response.choices[0].message.content.strip()
    # Some models wrap the JSON in a markdown code fence despite the
    # "Return ONLY a JSON object" instruction; strip it defensively.
    raw = raw.replace("```json", "").replace("```", "").strip()
    effects = json.loads(raw)

    return effects, usage, elapsed

gpt_effects, gpt_usage, gpt_time = call_gpt(prompt)
print(f"LLM - time: {gpt_time:.4f}s")
print(f"Token: input={gpt_usage['input_tokens']}, "
      f"output={gpt_usage['output_tokens']}, "
      f"total={gpt_usage['total_tokens']}")
print(json.dumps(gpt_effects, indent=2))

INFO:openai._base_client:Retrying request to /chat/completions in 0.388202 seconds
INFO:httpx:HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"


LLM - time: 670.3290s
Token: input=27745, output=48, total=27793
{
  "TV": 0.0,
  "TE": 0.0,
  "SE": 0.0,
  "DE": 0.0,
  "IE": 0.0
}


## 7. Comparing FairMind vs LLM

Computes the absolute and relative error between the two sets of effects, one row per metric.

In [69]:
def compute_discrepancies(ground_truth: dict, gpt_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt  = ground_truth.get(effect, float("nan"))
        gpt = float(gpt_effects.get(effect, float("nan")))
        abs_err = abs(gt - gpt)
        # Avoid computing the relative error when the reference value is effectively zero.
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect":      effect,
            "fairmind":    round(gt,  6),
            "gpt":         round(gpt, 6),
            "abs_error":   round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)

discrepancies = compute_discrepancies(ground_truth, gpt_effects)
print(discrepancies.to_string(index=False))

effect  fairmind  gpt  abs_error  rel_error_%
    TV  0.194470  0.0   0.194470        100.0
    TE  0.183161  0.0   0.183161        100.0
    SE  0.011309  0.0   0.011309        100.0
    DE  0.137049  0.0   0.137049        100.0
    IE -0.046112  0.0   0.046112        100.0


## 8. Saving results to disk

Writes configuration, both sets of effects, the discrepancy table, token usage and timing to a timestamped JSON file inside of `benchmark_results/`.

In [70]:
def save_results(config, ground_truth, gpt_effects, discrepancies, usage, fairmind_time, gpt_time):
    import os, datetime
    os.makedirs("benchmark_results", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/{config['dataset_name']}_{ts}.json"

    out = {
        "dataset":       config["dataset_name"],
        "config":        {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind":      ground_truth,
        "gpt":           gpt_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage":   usage,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "gpt_seconds":      round(gpt_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved: {fname}")

save_results(CONFIG, ground_truth, gpt_effects, discrepancies, gpt_usage, fairmind_time, gpt_time)

Saved: benchmark_results/adult_20260621_223117_SEfixed.json


## Update: the SE ground truth has been corrected

**What was wrong.** `run_fairmind()` originally computed the SE ground truth as `spurious_effect(bn, target, protected, x0)` - a single-argument quantity equal to `P(y|x) - P(y|do(x))` for one group. The prompt sent to the LLM, on the other hand, asked for `SE = TV - TE`, the two-argument identity from Eq. 3 of Plečko and Bareinboim (2024). These are two different mathematical objects that happen to share the name spurious effect.

**The numbers.** With the original (wrong) function: `SE = -0.007296`. With `SE = TV - TE` computed directly from the same fitted network: `SE = 0.011309` - opposite sign, different order of magnitude. `run_fairmind()` above now computes `tv` and `te` once and derives `SE` from them, matching the identity actually stated in the prompt.

**Why this is being fixed here.** This mismatch was found later, but the fix is applied here too so this notebook's own ground truth is internally consistent with the prompt it sends to the LLM. The mistake itself, and how it was found, are not erased: they are recorded in this cell and in the documentation.